# Model evaluation

Uses generated synthetic prices for software research only; these are not actual market data.

Models share one chronological holdout. Use validation data for model selection; avoid repeatedly tuning against the final test window.


In [ ]:
from pathlib import Path
import sys
repo = Path.cwd()
while not (repo / "backend").is_dir() and repo != repo.parent:
    repo = repo.parent
sys.path.insert(0, str(repo / "backend"))

import pickle
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from trading.data.synthetic import generate_synthetic_ohlcv
from trading.features.feature_pipeline import engineer_features
from trading.ml.targets import next_period_direction
from trading.ml.trainer import SignalModelTrainer
names = ["return_5", "return_20", "rsi_14", "macd_line", "atr_14",
         "rolling_volatility_20", "volume_change_1"]
frame = engineer_features(generate_synthetic_ohlcv(periods=1200, frequency="1D", seed=23))
frame["target"] = next_period_direction(frame.close)
frame = frame.dropna(subset=[*names, "target"]).reset_index(drop=True)
train_end, test_start = int(len(frame) * .6), int(len(frame) * .8)
train, test = frame.iloc[:train_end - 1], frame.iloc[test_start:]
truth = test.target.astype(int).to_numpy()
rows, trainer = [], SignalModelTrainer()
for kind in ("logistic_regression", "random_forest"):
    artifact = trainer.fit_artifact(
        features={name: train[name].astype(float).tolist() for name in names},
        labels=train.target.astype(int).tolist(), model_type=kind)
    model = pickle.loads(artifact)  # Trusted local artifact generated above.
    matrix = test[names].to_numpy(dtype=float)
    predicted = model.predict(matrix)
    probability = model.predict_proba(matrix)[:, list(model.classes_).index(1)]
    rows.append({"model": kind, "accuracy": accuracy_score(truth, predicted),
                 "precision": precision_score(truth, predicted, zero_division=0),
                 "recall": recall_score(truth, predicted, zero_division=0),
                 "f1": f1_score(truth, predicted, zero_division=0),
                 "roc_auc": roc_auc_score(truth, probability)
                 if np.unique(truth).size == 2 else None})
pd.DataFrame(rows).set_index("model")
